<a href="https://colab.research.google.com/github/hungryrobot1/memetic-time-machine/blob/main/notebooks/memetic_geometry_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Memetic Geometry in Activation Space

**Exploring whether neural network activations encode memetic potency**

This notebook investigates whether language model activations contain geometric signatures that correlate with memetic fitness—the tendency of phrases to propagate, mutate, and persist in cultural transmission.

We use data from the [MemeTracker project](https://snap.stanford.edu/memetracker/) (Leskovec, Backstrom, Kleinberg, KDD 2009), which tracked phrase propagation across 90 million news articles and blog posts during the 2008 US Presidential Election.

**Key hypotheses:**
1. Strong memes occupy *regions* rather than points—their variants span coherent, bounded subspaces
2. Hull area correlates with memetic fitness (log frequency)
3. Strong meme hulls have characteristic shapes (elongated, parallel axes)
4. Weak memes either collapse to tight clusters OR fragment into incoherent scatter

---

## 1. Setup & Dependencies

In [79]:
# Install dependencies
!pip install -q transformer_lens einops jaxtyping
!pip install -q umap-learn plotly scikit-learn

In [80]:
import torch
import numpy as np
import json
from typing import Dict, List, Tuple, Optional
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial import ConvexHull
from scipy import stats
import umap
import pandas as pd

from transformer_lens import HookedTransformer

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## 2. Mount Google Drive & Load Data

In [81]:
from pathlib import Path
import sys, urllib.request

RAW = ("https://raw.githubusercontent.com/hungryrobot1/"
       "memetic-time-machine/main/data/memes_curated.json")

def find_upward(name, start=None, levels=5):
    """Local runs: walk up looking for data/<name>."""
    d = (start or Path.cwd()).resolve()
    for _ in range(levels):
        hit = d / "data" / name
        if hit.exists():
            return hit
        d = d.parent
    return None

CORPUS = find_upward("memes_curated.json")

if CORPUS is None:
    # Bare runtime (Colab): pull the corpus straight from the repo.
    CORPUS = Path("data/memes_curated.json")
    CORPUS.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(RAW, CORPUS)

print(f"corpus: {CORPUS.resolve()} ({CORPUS.stat().st_size:,} bytes)")
DATA_FILE = CORPUS          # keep downstream cells unchanged


corpus: /content/data/memes_curated.json (28,728 bytes)


In [82]:
# Load the curated meme corpus
with open(DATA_FILE, 'r') as f:
    MEME_DATA = json.load(f)

print(f"Source: {MEME_DATA['metadata']['source']}")
print(f"Date range: {MEME_DATA['metadata']['date_range']}")
print(f"Strong memes: {len(MEME_DATA['strong_memes'])}")
print(f"Weak memes: {len(MEME_DATA['weak_memes'])}")

# Show summary
print("\n=== Strong Memes ===")
for m in MEME_DATA['strong_memes']:
    print(f"  {m['id'][:30]:30s} freq={m['total_frequency']:>6,} variants={len(m['variants'])}")

print("\n=== Weak Memes ===")
for m in MEME_DATA['weak_memes']:
    print(f"  {m['id'][:30]:30s} freq={m['total_frequency']:>6,} variants={len(m['variants'])}")

Source: MemeTracker (SNAP Stanford)
Date range: August 2008 - October 2008
Strong memes: 5
Weak memes: 5

=== Strong Memes ===
  joe_the_plumber                freq=19,393 variants=6
  you_can_put_lipstick_on_a_pig  freq=12,851 variants=15
  the_chant_is_drill_baby_drill  freq= 7,616 variants=5
  they_are_too_big_to_fail       freq= 5,919 variants=4
  our_national_leaders_are_sendi freq= 3,817 variants=15

=== Weak Memes ===
  well_you_know_that_mr_obama_is freq= 1,700 variants=15
  our_entire_economy_is_in_dange freq= 1,680 variants=11
  this_election_is_not_about_iss freq= 1,503 variants=15
  two_wars_a_planet_in_peril_the freq= 1,370 variants=15
  it_s_been_a_long_time_coming_b freq= 1,256 variants=8


In [83]:
def flatten_corpus(meme_data: dict) -> Tuple[List[str], List[str], List[str], List[int]]:
    """
    Flatten the nested meme corpus into parallel lists.

    Returns:
        phrases: List of all phrase variants
        labels: List of 'strong' or 'weak' for each phrase
        meme_ids: List of parent meme ID for each phrase
        frequencies: List of total cluster frequency for each phrase
    """
    phrases = []
    labels = []
    meme_ids = []
    frequencies = []

    for meme in meme_data['strong_memes']:
        for variant in meme['variants']:
            phrases.append(variant)
            labels.append('strong')
            meme_ids.append(meme['id'])
            frequencies.append(meme['total_frequency'])

    for meme in meme_data['weak_memes']:
        for variant in meme['variants']:
            phrases.append(variant)
            labels.append('weak')
            meme_ids.append(meme['id'])
            frequencies.append(meme['total_frequency'])

    return phrases, labels, meme_ids, frequencies

phrases, labels, meme_ids, frequencies = flatten_corpus(MEME_DATA)
print(f"Total phrases: {len(phrases)}")
print(f"  Strong: {labels.count('strong')}")
print(f"  Weak: {labels.count('weak')}")
print(f"  Unique meme families: {len(set(meme_ids))}")

Total phrases: 109
  Strong: 45
  Weak: 64
  Unique meme families: 10


## 3. Load Model

In [84]:
# Load Pythia-2.8B by default
# Any model supported by TransformerLens will work
MODEL_NAME = "gpt2-small"

print(f"Loading {MODEL_NAME}...")
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    device=device,
    dtype=torch.float16 if device == "cuda" else torch.float32
)
print(f"Model loaded. Layers: {model.cfg.n_layers}, d_model: {model.cfg.d_model}")

Loading gpt2-small...


/tmp/ipykernel_781/3776772221.py:6: DeprecationWarning:

HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer
Model loaded. Layers: 12, d_model: 768


## 4. Extract Activations

In [85]:
def get_phrase_activations(
    model: HookedTransformer,
    phrases: List[str],
    layers: List[int]
) -> Dict[int, torch.Tensor]:
    """
    Extract mean-pooled residual stream activations for each phrase.

    Args:
        model: HookedTransformer model
        phrases: List of text phrases
        layers: Which layers to extract from

    Returns:
        Dict mapping layer index to tensor of shape (n_phrases, d_model)
    """
    layer_activations = {layer: [] for layer in layers}

    for phrase in phrases:
        # Run with cache
        _, cache = model.run_with_cache(phrase, prepend_bos=True)

        for layer in layers:
            # Get residual stream at this layer
            # Shape: (1, seq_len, d_model)
            resid = cache[f"blocks.{layer}.hook_resid_post"]

            # Mean pool across sequence (excluding BOS)
            mean_act = resid[0, 1:, :].mean(dim=0)
            layer_activations[layer].append(mean_act.cpu())

    # Stack into tensors
    for layer in layers:
        layer_activations[layer] = torch.stack(layer_activations[layer])

    return layer_activations

In [86]:
# Extract activations at multiple layers
# Based on prior experiments, layer 8 shows best separation
LAYERS_TO_ANALYZE = [0, 4, 8, 11]

print(f"Extracting activations for {len(phrases)} phrases...")
print(f"Layers: {LAYERS_TO_ANALYZE}")

layer_activations = get_phrase_activations(model, phrases, LAYERS_TO_ANALYZE)

print(f"\nActivation shapes:")
for layer, acts in layer_activations.items():
    print(f"  Layer {layer}: {acts.shape}")

Extracting activations for 109 phrases...
Layers: [0, 4, 8, 11]

Activation shapes:
  Layer 0: torch.Size([109, 768])
  Layer 4: torch.Size([109, 768])
  Layer 8: torch.Size([109, 768])
  Layer 11: torch.Size([109, 768])


## 5. UMAP Projection & Layer Selection

In [87]:
def compute_umap_embedding(
    activations: torch.Tensor,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    random_state: int = 42
) -> np.ndarray:
    """
    Compute 2D UMAP embedding from high-dimensional activations.
    """
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=2,
        metric='cosine',
        random_state=random_state
    )
    embedding = reducer.fit_transform(activations.numpy())
    return embedding


def analyze_layer_separation(
    activations: torch.Tensor,
    labels: List[str]
) -> Tuple[np.ndarray, float]:
    """
    Compute UMAP embedding and silhouette score for strong/weak separation.
    """
    embedding = compute_umap_embedding(activations)

    # Convert labels to numeric for silhouette
    label_map = {'strong': 1, 'weak': 0}
    numeric_labels = [label_map[l] for l in labels]

    sil_score = silhouette_score(embedding, numeric_labels)

    return embedding, sil_score

In [88]:
# Find best layer for strong/weak separation
print("Analyzing layer separation (silhouette scores):")
print("-" * 40)

layer_results = {}
for layer, acts in layer_activations.items():
    embedding, sil_score = analyze_layer_separation(acts, labels)
    layer_results[layer] = {
        'embedding': embedding,
        'silhouette': sil_score
    }
    print(f"Layer {layer:2d}: silhouette = {sil_score:.4f}")

# Select best layer
BEST_LAYER = max(layer_results.keys(), key=lambda k: layer_results[k]['silhouette'])
print(f"\n→ Best layer: {BEST_LAYER} (silhouette = {layer_results[BEST_LAYER]['silhouette']:.4f})")

Analyzing layer separation (silhouette scores):
----------------------------------------


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Layer  0: silhouette = 0.2493


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Layer  4: silhouette = 0.1811


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Layer  8: silhouette = 0.1626


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Layer 11: silhouette = 0.0775

→ Best layer: 0 (silhouette = 0.2493)


In [89]:
# Use best layer for subsequent analysis
embedding = layer_results[BEST_LAYER]['embedding']

# Build dataframe for visualization
df = pd.DataFrame({
    'umap_1': embedding[:, 0],
    'umap_2': embedding[:, 1],
    'label': labels,
    'meme_id': meme_ids,
    'frequency': frequencies,
    'log_freq': np.log10(frequencies),
    'phrase': [p[:60] + '...' if len(p) > 60 else p for p in phrases]
})

df.head()

,umap_1,umap_2,label,meme_id,frequency,log_freq,phrase
0,7.690475,-0.337048,strong,joe_the_plumber,19393,4.287645,joe the plumber
1,7.794692,-0.396079,strong,joe_the_plumber,19393,4.287645,i'm joe the plumber
2,7.969995,-0.575296,strong,joe_the_plumber,19393,4.287645,joe the biden
3,7.575508,-0.644126,strong,joe_the_plumber,19393,4.287645,not the plumber
4,7.942803,-0.225811,strong,joe_the_plumber,19393,4.287645,i'm joe not the plumber joe the biden


In [90]:
# Visualize by label (strong vs weak)
fig = px.scatter(
    df, x='umap_1', y='umap_2',
    color='label',
    color_discrete_map={'strong': 'red', 'weak': 'blue'},
    hover_data=['phrase', 'meme_id', 'frequency'],
    title=f'Meme Activations in UMAP Space (Layer {BEST_LAYER})',
    labels={'umap_1': 'UMAP 1', 'umap_2': 'UMAP 2'}
)
fig.update_traces(marker=dict(size=10, opacity=0.7))
fig.show()

In [91]:
# Visualize by meme family
fig = px.scatter(
    df, x='umap_1', y='umap_2',
    color='meme_id',
    symbol='label',
    symbol_map={'strong': 'circle', 'weak': 'x'},
    hover_data=['phrase', 'frequency'],
    title=f'Meme Families in UMAP Space (Layer {BEST_LAYER})',
    labels={'umap_1': 'UMAP 1', 'umap_2': 'UMAP 2'}
)
fig.update_traces(marker=dict(size=10, opacity=0.8))
fig.show()

## 6. Convex Hull Analysis

**Core hypothesis**: Memetically fit phrases span *regions* rather than collapsing to points. We test this by computing convex hulls around each meme family's variants in UMAP space.

**Predictions**:
1. Hull area correlates positively with log(frequency)
2. Strong memes have larger, more regular hulls
3. Weak memes either collapse (tiny hull) or fragment (disjoint points)
4. Hulls are elongated (high eccentricity), not circular

In [92]:
def compute_hull_metrics(points: np.ndarray) -> Optional[dict]:
    """
    Compute convex hull and derived metrics for a set of 2D points.

    Returns None if fewer than 3 points (can't form a hull).

    Metrics:
    - area: Hull area in UMAP units
    - perimeter: Hull perimeter
    - n_vertices: Number of hull vertices (complexity)
    - centroid: Geometric center of hull
    - eccentricity: Ratio of major to minor axis (from bounding ellipse)
    - compactness: 4*pi*area / perimeter^2 (1 = circle, <1 = elongated)
    """
    if len(points) < 3:
        return None

    # Handle collinear points (hull would be degenerate)
    try:
        hull = ConvexHull(points)
    except:
        return None

    # Basic metrics
    area = hull.volume  # In 2D, 'volume' is area

    # Compute perimeter
    hull_points = points[hull.vertices]
    perimeter = 0
    for i in range(len(hull.vertices)):
        p1 = hull_points[i]
        p2 = hull_points[(i + 1) % len(hull.vertices)]
        perimeter += np.linalg.norm(p2 - p1)

    # Centroid (mean of all points, not just hull vertices)
    centroid = points.mean(axis=0)

    # Compactness (isoperimetric quotient)
    compactness = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

    # Eccentricity via PCA
    centered = points - centroid
    cov = np.cov(centered.T)
    eigenvalues = np.linalg.eigvalsh(cov)
    eigenvalues = np.sort(eigenvalues)[::-1]  # Descending

    # Eccentricity: ratio of major to minor axis (sqrt of eigenvalue ratio)
    if eigenvalues[1] > 1e-10:
        eccentricity = np.sqrt(eigenvalues[0] / eigenvalues[1])
    else:
        eccentricity = np.inf  # Degenerate (line)

    return {
        'hull': hull,
        'area': area,
        'perimeter': perimeter,
        'n_vertices': len(hull.vertices),
        'centroid': centroid,
        'compactness': compactness,
        'eccentricity': eccentricity,
        'eigenvalues': eigenvalues
    }

In [93]:
def analyze_meme_hulls(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute convex hull metrics for each meme family.
    """
    results = []

    for meme_id in df['meme_id'].unique():
        meme_df = df[df['meme_id'] == meme_id]
        points = meme_df[['umap_1', 'umap_2']].values

        label = meme_df['label'].iloc[0]
        frequency = meme_df['frequency'].iloc[0]
        n_variants = len(meme_df)

        metrics = compute_hull_metrics(points)

        result = {
            'meme_id': meme_id,
            'label': label,
            'frequency': frequency,
            'log_freq': np.log10(frequency),
            'n_variants': n_variants,
            'points': points
        }

        if metrics:
            result.update({
                'hull': metrics['hull'],
                'area': metrics['area'],
                'perimeter': metrics['perimeter'],
                'n_vertices': metrics['n_vertices'],
                'centroid': metrics['centroid'],
                'compactness': metrics['compactness'],
                'eccentricity': metrics['eccentricity']
            })
        else:
            result.update({
                'hull': None,
                'area': 0,
                'perimeter': 0,
                'n_vertices': 0,
                'centroid': points.mean(axis=0) if len(points) > 0 else np.array([0, 0]),
                'compactness': np.nan,
                'eccentricity': np.nan
            })

        results.append(result)

    return pd.DataFrame(results)

In [94]:
# Compute hull metrics for all meme families
hull_df = analyze_meme_hulls(df)

print("Hull Metrics by Meme Family:")
print("=" * 80)
display_cols = ['meme_id', 'label', 'frequency', 'n_variants', 'area', 'eccentricity', 'compactness']
print(hull_df[display_cols].to_string(index=False))

Hull Metrics by Meme Family:
                       meme_id  label  frequency  n_variants     area  eccentricity  compactness
               joe_the_plumber strong      19393           6 0.138703      2.122893     0.603334
 you_can_put_lipstick_on_a_pig strong      12851          15 0.440269      1.264916     0.846024
 the_chant_is_drill_baby_drill strong       7616           5 3.405569      2.886053     0.475659
      they_are_too_big_to_fail strong       5919           4 0.040068      3.746492     0.341091
our_national_leaders_are_sendi strong       3817          15 3.336599      5.464744     0.366591
well_you_know_that_mr_obama_is   weak       1700          15 1.919997      8.675869     0.252778
our_entire_economy_is_in_dange   weak       1680          11 1.686683      9.800678     0.261781
this_election_is_not_about_iss   weak       1503          15 1.479809      7.448889     0.317608
two_wars_a_planet_in_peril_the   weak       1370          15 2.648740      4.166963     0.399536
i

In [95]:
# Visualize hulls overlaid on UMAP
fig = go.Figure()

# Color maps
strong_colors = px.colors.qualitative.Set1
weak_colors = px.colors.qualitative.Set2

strong_idx = 0
weak_idx = 0

for _, row in hull_df.iterrows():
    points = row['points']
    hull = row['hull']
    meme_id = row['meme_id']
    label = row['label']

    # Assign color
    if label == 'strong':
        color = strong_colors[strong_idx % len(strong_colors)]
        strong_idx += 1
    else:
        color = weak_colors[weak_idx % len(weak_colors)]
        weak_idx += 1

    # Plot points
    fig.add_trace(go.Scatter(
        x=points[:, 0],
        y=points[:, 1],
        mode='markers',
        marker=dict(
            size=10 if label == 'strong' else 8,
            color=color,
            symbol='circle' if label == 'strong' else 'x',
            opacity=0.8
        ),
        name=f"{meme_id[:20]} ({label[0].upper()})",
        legendgroup=meme_id,
        hovertemplate=f"{meme_id}<br>freq={row['frequency']:,}<extra></extra>"
    ))

    # Plot hull if exists
    if hull is not None:
        hull_points = points[hull.vertices]
        # Close the polygon
        hull_points = np.vstack([hull_points, hull_points[0]])

        fig.add_trace(go.Scatter(
            x=hull_points[:, 0],
            y=hull_points[:, 1],
            mode='lines',
            line=dict(color=color, width=2, dash='solid' if label == 'strong' else 'dash'),
            fill='toself',
            fillcolor=color,
            opacity=0.15,
            name=f"{meme_id[:20]} hull",
            legendgroup=meme_id,
            showlegend=False,
            hoverinfo='skip'
        ))

fig.update_layout(
    title=f'Convex Hulls of Meme Families (Layer {BEST_LAYER})',
    xaxis_title='UMAP 1',
    yaxis_title='UMAP 2',
    width=900,
    height=700,
    legend=dict(x=1.02, y=1)
)
fig.show()

## 7. Statistical Tests

In [96]:
# Test 1: Hull area vs log frequency correlation
# Only include memes with valid hulls (3+ variants)
valid_hulls = hull_df[hull_df['area'] > 0].copy()

if len(valid_hulls) >= 3:
    r, p = stats.pearsonr(valid_hulls['log_freq'], valid_hulls['area'])

    print("Test 1: Hull Area vs Log(Frequency)")
    print(f"  Pearson r = {r:.4f}")
    print(f"  p-value = {p:.4f}")
    print(f"  n = {len(valid_hulls)}")
    print(f"  Interpretation: {'Significant' if p < 0.05 else 'Not significant'} correlation")

    # Scatter plot
    fig = px.scatter(
        valid_hulls,
        x='log_freq',
        y='area',
        color='label',
        color_discrete_map={'strong': 'red', 'weak': 'blue'},
        hover_data=['meme_id', 'n_variants'],
        title=f'Hull Area vs Log(Frequency) (r={r:.3f}, p={p:.3f})',
        labels={'log_freq': 'Log₁₀(Frequency)', 'area': 'Hull Area (UMAP²)'}
    )

    # Add trend line
    slope, intercept = np.polyfit(valid_hulls['log_freq'], valid_hulls['area'], 1)
    x_range = np.linspace(valid_hulls['log_freq'].min(), valid_hulls['log_freq'].max(), 100)
    fig.add_trace(go.Scatter(
        x=x_range,
        y=slope * x_range + intercept,
        mode='lines',
        line=dict(color='gray', dash='dash'),
        name='Trend'
    ))

    fig.show()
else:
    print("Not enough valid hulls for correlation test")

Test 1: Hull Area vs Log(Frequency)
  Pearson r = -0.3876
  p-value = 0.2684
  n = 10
  Interpretation: Not significant correlation


In [97]:
# Test 2: Strong vs Weak hull area comparison
strong_areas = hull_df[hull_df['label'] == 'strong']['area'].values
weak_areas = hull_df[hull_df['label'] == 'weak']['area'].values

print("\nTest 2: Strong vs Weak Hull Areas")
print(f"  Strong memes: mean area = {strong_areas.mean():.4f} (n={len(strong_areas)})")
print(f"  Weak memes: mean area = {weak_areas.mean():.4f} (n={len(weak_areas)})")

# Mann-Whitney U test (non-parametric, small sample)
if len(strong_areas) >= 2 and len(weak_areas) >= 2:
    u_stat, p_mw = stats.mannwhitneyu(strong_areas, weak_areas, alternative='greater')
    print(f"  Mann-Whitney U = {u_stat:.1f}")
    print(f"  p-value (one-tailed, strong > weak) = {p_mw:.4f}")

# Box plot comparison
fig = px.box(
    hull_df,
    x='label',
    y='area',
    color='label',
    color_discrete_map={'strong': 'red', 'weak': 'blue'},
    points='all',
    title='Hull Area Distribution: Strong vs Weak Memes'
)
fig.show()


Test 2: Strong vs Weak Hull Areas
  Strong memes: mean area = 1.4722 (n=5)
  Weak memes: mean area = 1.8374 (n=5)
  Mann-Whitney U = 10.0
  p-value (one-tailed, strong > weak) = 0.7262


In [98]:
# Test 3: Eccentricity comparison
print("\nTest 3: Hull Eccentricity (elongation)")

valid_ecc = hull_df[hull_df['eccentricity'].notna() & (hull_df['eccentricity'] < np.inf)]

strong_ecc = valid_ecc[valid_ecc['label'] == 'strong']['eccentricity'].values
weak_ecc = valid_ecc[valid_ecc['label'] == 'weak']['eccentricity'].values

print(f"  Strong memes: mean eccentricity = {strong_ecc.mean():.4f} (n={len(strong_ecc)})")
print(f"  Weak memes: mean eccentricity = {weak_ecc.mean():.4f} (n={len(weak_ecc)})")
print(f"  (Eccentricity > 1 means elongated; = 1 means circular)")

if len(valid_ecc) > 0:
    fig = px.scatter(
        valid_ecc,
        x='area',
        y='eccentricity',
        color='label',
        color_discrete_map={'strong': 'red', 'weak': 'blue'},
        size='n_variants',
        hover_data=['meme_id'],
        title='Hull Shape: Area vs Eccentricity',
        labels={'area': 'Hull Area', 'eccentricity': 'Eccentricity (major/minor axis)'}
    )
    fig.add_hline(y=1, line_dash='dash', line_color='gray', annotation_text='circular')
    fig.show()


Test 3: Hull Eccentricity (elongation)
  Strong memes: mean eccentricity = 3.0970 (n=5)
  Weak memes: mean eccentricity = 7.0840 (n=5)
  (Eccentricity > 1 means elongated; = 1 means circular)


## 8. Detailed Family Analysis

Examine specific meme families to understand their geometric structure.

In [99]:
def plot_single_family(meme_id: str, df: pd.DataFrame, hull_df: pd.DataFrame):
    """
    Create detailed visualization of a single meme family.
    """
    meme_df = df[df['meme_id'] == meme_id]
    hull_row = hull_df[hull_df['meme_id'] == meme_id].iloc[0]

    fig = go.Figure()

    # Plot hull
    if hull_row['hull'] is not None:
        hull = hull_row['hull']
        points = hull_row['points']
        hull_points = points[hull.vertices]
        hull_points = np.vstack([hull_points, hull_points[0]])

        fig.add_trace(go.Scatter(
            x=hull_points[:, 0],
            y=hull_points[:, 1],
            mode='lines',
            fill='toself',
            fillcolor='rgba(100, 100, 255, 0.2)',
            line=dict(color='blue', width=2),
            name='Convex Hull'
        ))

    # Plot points with phrase labels
    fig.add_trace(go.Scatter(
        x=meme_df['umap_1'],
        y=meme_df['umap_2'],
        mode='markers+text',
        marker=dict(size=12, color='red'),
        text=[str(i) for i in range(len(meme_df))],
        textposition='top center',
        name='Variants',
        hovertext=meme_df['phrase']
    ))

    # Plot centroid
    centroid = hull_row['centroid']
    fig.add_trace(go.Scatter(
        x=[centroid[0]],
        y=[centroid[1]],
        mode='markers',
        marker=dict(size=15, color='green', symbol='x'),
        name='Centroid'
    ))

    fig.update_layout(
        title=f"'{meme_id}' Family Structure<br>" +
              f"Area={hull_row['area']:.3f}, Ecc={hull_row['eccentricity']:.2f}, n={hull_row['n_variants']}",
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        width=700,
        height=600
    )
    fig.show()

    # Print variant list
    print(f"\nVariants of '{meme_id}':")
    for i, phrase in enumerate(meme_df['phrase'].values):
        print(f"  [{i}] {phrase}")

In [100]:
# Analyze the largest strong meme
largest_strong = hull_df[hull_df['label'] == 'strong'].sort_values('area', ascending=False).iloc[0]['meme_id']
print(f"Largest strong meme hull: {largest_strong}")
plot_single_family(largest_strong, df, hull_df)

Largest strong meme hull: the_chant_is_drill_baby_drill



Variants of 'the_chant_is_drill_baby_drill':
  [0] drill baby drill
  [1] the chant is drill baby drill
  [2] the chant is drill baby drill and that's what we hear all ac...
  [3] the chant is
  [4] and that's what we hear all across this country in our ralli...


In [101]:
# Analyze the "obama is a muslim" cluster for bimodality
# This cluster may contain both the accusation AND the rebuttal
obama_cluster = [m for m in hull_df['meme_id'] if 'obama' in m.lower()]
if obama_cluster:
    print(f"Analyzing: {obama_cluster[0]}")
    plot_single_family(obama_cluster[0], df, hull_df)

Analyzing: well_you_know_that_mr_obama_is



Variants of 'well_you_know_that_mr_obama_is':
  [0] well you know that mr obama is a muslim
  [1] is something wrong
  [2] well you know
  [3] is there something wrong with being a muslim in this country
  [4] always been a christian
  [5] the really right answer is what if he is
  [6] is there something wrong
  [7] he's always been a christian but the really right answer is ...
  [8] you know that
  [9] what if he is is there something wrong with being a muslim i...
  [10] what if he is
  [11] well the correct answer is he is not a muslim he's a christi...
  [12] he has always been a christian but the really right answer i...
  [13] he's not a muslim he's a christian
  [14] the correct answer is he is not a muslim he's a christian he...


## 9. Within-Cluster Similarity Analysis

Cosine similarity in high-dimensional space (before UMAP projection).

In [102]:
def analyze_within_cluster_similarity(
    activations: torch.Tensor,
    meme_ids: List[str],
    phrases: List[str]
) -> pd.DataFrame:
    """
    Compute within-cluster cosine similarity in original activation space.
    """
    X = activations.numpy()
    results = []

    for meme_id in set(meme_ids):
        indices = [i for i, m in enumerate(meme_ids) if m == meme_id]
        if len(indices) < 2:
            continue

        cluster_acts = X[indices]
        sim_matrix = cosine_similarity(cluster_acts)

        # Average similarity (excluding diagonal)
        n = len(indices)
        avg_sim = (sim_matrix.sum() - n) / (n * n - n)

        # Min and max off-diagonal similarity
        off_diag = sim_matrix[~np.eye(n, dtype=bool)]
        min_sim = off_diag.min()
        max_sim = off_diag.max()

        results.append({
            'meme_id': meme_id,
            'n_variants': n,
            'avg_similarity': avg_sim,
            'min_similarity': min_sim,
            'max_similarity': max_sim,
            'similarity_range': max_sim - min_sim
        })

    return pd.DataFrame(results)

In [103]:
# Compute similarity metrics
sim_df = analyze_within_cluster_similarity(
    layer_activations[BEST_LAYER],
    meme_ids,
    phrases
)

# Merge with hull data
combined_df = hull_df.merge(sim_df, on='meme_id')

print("Combined Hull + Similarity Metrics:")
print("=" * 90)
display_cols = ['meme_id', 'label', 'frequency', 'area', 'avg_similarity', 'similarity_range']
print(combined_df[display_cols].sort_values('frequency', ascending=False).to_string(index=False))

Combined Hull + Similarity Metrics:
                       meme_id  label  frequency     area  avg_similarity  similarity_range
               joe_the_plumber strong      19393 0.138703        0.959916          0.065580
 you_can_put_lipstick_on_a_pig strong      12851 0.440269        0.978743          0.047339
 the_chant_is_drill_baby_drill strong       7616 3.405569        0.913373          0.139628
      they_are_too_big_to_fail strong       5919 0.040068        0.984341          0.022379
our_national_leaders_are_sendi strong       3817 3.336599        0.940592          0.141908
well_you_know_that_mr_obama_is   weak       1700 1.919997        0.941023          0.143320
our_entire_economy_is_in_dange   weak       1680 1.686683        0.959398          0.102331
this_election_is_not_about_iss   weak       1503 1.479809        0.964253          0.070142
two_wars_a_planet_in_peril_the   weak       1370 2.648740        0.955582          0.132518
it_s_been_a_long_time_coming_b   weak       

In [104]:
# Test: Do strong memes have LOWER similarity (more spread)?
print("\nWithin-cluster similarity comparison:")

strong_sim = combined_df[combined_df['label'] == 'strong']['avg_similarity'].values
weak_sim = combined_df[combined_df['label'] == 'weak']['avg_similarity'].values

print(f"  Strong memes: mean similarity = {strong_sim.mean():.4f}")
print(f"  Weak memes: mean similarity = {weak_sim.mean():.4f}")

# Hypothesis: Strong memes have LOWER similarity (more productive variation)
if len(strong_sim) >= 2 and len(weak_sim) >= 2:
    u_stat, p_mw = stats.mannwhitneyu(strong_sim, weak_sim, alternative='less')
    print(f"  Mann-Whitney U = {u_stat:.1f}")
    print(f"  p-value (one-tailed, strong < weak) = {p_mw:.4f}")
    print(f"  → {'Supports' if p_mw < 0.1 else 'Does not support'} hypothesis that strong memes have more internal variation")


Within-cluster similarity comparison:
  Strong memes: mean similarity = 0.9554
  Weak memes: mean similarity = 0.9544
  Mann-Whitney U = 14.0
  p-value (one-tailed, strong < weak) = 0.6548
  → Does not support hypothesis that strong memes have more internal variation


In [105]:
# Scatter: Area vs Similarity (inverse relationship expected for strong memes)
fig = px.scatter(
    combined_df,
    x='avg_similarity',
    y='area',
    color='label',
    color_discrete_map={'strong': 'red', 'weak': 'blue'},
    size='n_variants_x',
    hover_data=['meme_id', 'frequency'],
    title='Hull Area vs Within-Cluster Similarity',
    labels={'avg_similarity': 'Avg Cosine Similarity', 'area': 'Hull Area'}
)
fig.show()

## 10. Summary & Conclusions

In [108]:
print("="*80)
print("SUMMARY: Memetic Geometry Experiment Results")
print("="*80)

print(f"\nDataset: {len(df)} phrase variants across {len(hull_df)} meme families")
print(f"Model: {MODEL_NAME}, Layer: {BEST_LAYER}")
print(f"Best layer silhouette score: {layer_results[BEST_LAYER]['silhouette']:.4f}")

print("\n--- Hull Area Analysis ---")
strong_area_mean = hull_df[hull_df['label'] == 'strong']['area'].mean()
weak_area_mean = hull_df[hull_df['label'] == 'weak']['area'].mean()
print(f"Strong memes mean hull area: {strong_area_mean:.4f}")
print(f"Weak memes mean hull area: {weak_area_mean:.4f}")
print(f"Ratio (strong/weak): {strong_area_mean/weak_area_mean:.2f}x" if weak_area_mean > 0 else "N/A")

print("\n--- Within-Cluster Similarity ---")
print(f"Strong memes mean similarity: {combined_df[combined_df['label']=='strong']['avg_similarity'].mean():.4f}")
print(f"Weak memes mean similarity: {combined_df[combined_df['label']=='weak']['avg_similarity'].mean():.4f}")

print("\n--- Key Findings ---")
print("Strong memes tend to have smaller areas, whereas weak memes disperse.")
print("This suggests a model of potent memes as attractors.")
print("The sample corpus is too small to be considered quantitatively decisive.")
print("--- Next Steps ---")
print("1. Test on larger corpus (more meme families)")
print("2. Analyze hull overlap between families")
print("3. Track temporal evolution of hull shape")
print("4. Test whether new variants land inside family hulls")

SUMMARY: Memetic Geometry Experiment Results

Dataset: 109 phrase variants across 10 meme families
Model: gpt2-small, Layer: 0
Best layer silhouette score: 0.2493

--- Hull Area Analysis ---
Strong memes mean hull area: 1.4722
Weak memes mean hull area: 1.8374
Ratio (strong/weak): 0.80x

--- Within-Cluster Similarity ---
Strong memes mean similarity: 0.9554
Weak memes mean similarity: 0.9544

--- Key Findings ---
Strong memes tend to have smaller areas, whereas weak memes disperse.
This suggests a model of potent memes as attractors.
The sample corpus is too small to be considered quantitatively decisive.
--- Next Steps ---
1. Test on larger corpus (more meme families)
2. Analyze hull overlap between families
3. Track temporal evolution of hull shape
4. Test whether new variants land inside family hulls


In [109]:
# Save results to Drive for later analysis
output_file = DATA_DIR / 'hull_analysis_results.json'

# Prepare serializable results
export_data = {
    'metadata': {
        'model': MODEL_NAME,
        'best_layer': BEST_LAYER,
        'silhouette_score': layer_results[BEST_LAYER]['silhouette'],
        'n_phrases': len(df),
        'n_families': len(hull_df)
    },
    'hull_metrics': combined_df.drop(columns=['hull', 'points', 'centroid'], errors='ignore').to_dict(orient='records'),
    'layer_silhouettes': {str(k): v['silhouette'] for k, v in layer_results.items()}
}

with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2, default=str)

print(f"Results saved to: {output_file}")

Results saved to: /content/drive/MyDrive/Colab Notebooks/memetic-time-machine/hull_analysis_results.json
